# UNO Game AI – Adversarial Search (Minimax + Expectimax)
### Three-Player Simplified UNO with Defensive and Offensive AI Strategies
---

## 1. Imports and Setup

In [ ]:
import random
import copy
import math
from collections import defaultdict

## 2. Card Class

In [ ]:
class Card:
    """
    Represents a single UNO card.
    color: 'Red', 'Blue', 'Green', 'Yellow'
    value: 0-9 or 'Skip'
    """
    def __init__(self, color, value):
        self.color = color
        self.value = value

    def is_skip(self):
        return self.value == 'Skip'

    def __repr__(self):
        return f"{self.color} {self.value}"

    def __eq__(self, other):
        return self.color == other.color and self.value == other.value

    def __hash__(self):
        return hash((self.color, self.value))

## 3. Deck Generator

In [ ]:
def generate_deck():
    """
    Generates and shuffles a simplified UNO deck.
    Contains:
      - Red, Blue, Green, Yellow cards numbered 0–9
      - One Skip card per color
    """
    colors = ['Red', 'Blue', 'Green', 'Yellow']
    deck = []

    for color in colors:
        for number in range(10):          # 0 through 9
            deck.append(Card(color, number))
        deck.append(Card(color, 'Skip'))   # one Skip per color

    random.shuffle(deck)
    return deck

# Quick test
test_deck = generate_deck()
print(f"Deck size: {len(test_deck)}")
print(f"First 5 cards: {test_deck[:5]}")

## 4. Legal Move Generator

In [ ]:
def get_valid_moves(hand, top_card):
    """
    Returns list of valid cards from 'hand' that can be played on 'top_card'.
    A card is valid if:
      - Same color as top_card, OR
      - Same number/value as top_card
    """
    valid = []
    for card in hand:
        if card.color == top_card.color or card.value == top_card.value:
            valid.append(card)
    return valid

# Quick test
top = Card('Red', 5)
hand = [Card('Red', 3), Card('Blue', 5), Card('Green', 7), Card('Yellow', 'Skip'), Card('Red', 'Skip')]
print(f"Top card: {top}")
print(f"Valid moves: {get_valid_moves(hand, top)}")

## 5. Game State and State Transition

In [ ]:
class GameState:
    """
    Represents the full game state:
      - hands[0], hands[1], hands[2]: card lists for P1, P2, P3
      - top_card: current top of discard pile
      - deck: remaining draw pile
      - current_player: 0, 1, or 2
    """
    def __init__(self, hands, top_card, deck, current_player=0):
        self.hands = hands          # list of 3 lists
        self.top_card = top_card
        self.deck = deck
        self.current_player = current_player

    def clone(self):
        return GameState(
            hands=[list(h) for h in self.hands],
            top_card=self.top_card,
            deck=list(self.deck),
            current_player=self.current_player
        )

    def is_terminal(self):
        """Game ends when any player has 0 cards."""
        return any(len(h) == 0 for h in self.hands)

    def winner(self):
        for i, h in enumerate(self.hands):
            if len(h) == 0:
                return i
        return None

    def __repr__(self):
        return (f"GameState(top={self.top_card}, "
                f"P1={len(self.hands[0])} cards, "
                f"P2={len(self.hands[1])} cards, "
                f"P3={len(self.hands[2])} cards, "
                f"deck={len(self.deck)} cards, "
                f"turn=P{self.current_player+1})")


def apply_move(state, move, skip_next=False):
    """
    Returns a NEW GameState after applying 'move' for the current player.
    move: a Card object to play, or None (meaning draw a card).
    """
    new_state = state.clone()
    cp = new_state.current_player
    players = [0, 1, 2]

    if move is None:
        # Draw a card
        if new_state.deck:
            drawn = new_state.deck.pop(0)
            new_state.hands[cp].append(drawn)
        # Turn passes to next player
        new_state.current_player = (cp + 1) % 3
    else:
        # Play the card
        new_state.hands[cp].remove(move)
        new_state.top_card = move

        if move.is_skip():
            # Skip the next player; move to player after that
            new_state.current_player = (cp + 2) % 3
        else:
            new_state.current_player = (cp + 1) % 3

    return new_state

## 6. Evaluation Function

### Formula:
$$\text{Score} = 50 - 5 \cdot C_{AI} + 2 \cdot C_{opp} + 3 \cdot S$$

- **C_AI**: Cards in the current player's hand (want to minimize)
- **C_opp**: Average cards held by the two opponents (want them to have more)
- **S**: Number of Skip cards in hand (strategic asset)

### Weight Tuning:
- **Defensive (P1 – Minimax)**: Penalize own cards more heavily; value Skip cards for blocking opponents.
- **Offensive (P2 – Expectimax)**: Reward aggressive card shedding; care less about opponents' counts.


In [ ]:
def evaluate(state, player_index, strategy='defensive'):
    """
    Evaluation function for a given player.
    strategy: 'defensive' (P1) or 'offensive' (P2/P3)
    
    Base formula: Score = 50 - 5*C_AI + 2*C_opp + 3*S
    
    Defensive tuning:
      - Increase penalty on own cards (w_ai = 6)
      - Increase reward for opponents having cards (w_opp = 3)
      - High skip value to block opponents (w_skip = 5)
    
    Offensive tuning:
      - Less penalty on own cards (w_ai = 4) — more risk-taking
      - Lower opponent weight (w_opp = 1) — focus on self
      - Moderate skip value (w_skip = 2)
    """
    own_cards = len(state.hands[player_index])
    opp_indices = [i for i in range(3) if i != player_index]
    avg_opp_cards = sum(len(state.hands[i]) for i in opp_indices) / 2.0
    skip_count = sum(1 for c in state.hands[player_index] if c.is_skip())

    if strategy == 'defensive':
        w_base = 50
        w_ai = 6       # heavier self-penalty
        w_opp = 3      # reward opponents having more cards
        w_skip = 5     # skips are very valuable defensively
    else:  # offensive
        w_base = 50
        w_ai = 4       # lighter self-penalty (aggressive shedding)
        w_opp = 1      # less concern about opponents
        w_skip = 2     # skips less critical offensively

    score = w_base - w_ai * own_cards + w_opp * avg_opp_cards + w_skip * skip_count

    # Large bonus/penalty for terminal states
    if state.is_terminal():
        winner = state.winner()
        if winner == player_index:
            score += 1000
        else:
            score -= 1000

    return score


# Test the evaluation function
deck = generate_deck()
hands = [deck[:5], deck[5:10], deck[10:15]]
top = deck[15]
rem_deck = deck[16:]
test_state = GameState(hands, top, rem_deck)

print(f"P1 defensive score: {evaluate(test_state, 0, 'defensive'):.2f}")
print(f"P2 offensive score: {evaluate(test_state, 1, 'offensive'):.2f}")
print(f"P3 defensive score: {evaluate(test_state, 2, 'defensive'):.2f}")

## 7. Minimax Algorithm (Defensive – Player 1 & 3)

In [ ]:
# Global tree log for visualization
minimax_tree_log = []

def minimax(state, depth, maximizing_player_index, current_turn_index,
            alpha=-math.inf, beta=math.inf, log_tree=False, tree_node=None):
    """
    Minimax with Alpha-Beta pruning.
    
    maximizing_player_index: which player we are optimizing for (0 or 2)
    current_turn_index: whose turn it currently is in this node
    
    Defensive assumption: all opponents play to MINIMIZE our score.
    """
    if tree_node is None:
        tree_node = {}

    if depth == 0 or state.is_terminal():
        val = evaluate(state, maximizing_player_index, 'defensive')
        tree_node['score'] = val
        tree_node['type'] = 'leaf'
        return val, None

    hand = state.hands[current_turn_index]
    valid_moves = get_valid_moves(hand, state.top_card)
    moves = valid_moves if valid_moves else [None]  # None = draw

    tree_node['player'] = f'P{current_turn_index + 1}'
    tree_node['top_card'] = str(state.top_card)
    tree_node['hand_size'] = len(hand)
    tree_node['children'] = []

    best_move = None

    if current_turn_index == maximizing_player_index:
        # MAX node
        tree_node['type'] = 'MAX'
        best_score = -math.inf
        for move in moves:
            child_state = apply_move(state, move)
            child_node = {}
            score, _ = minimax(child_state, depth - 1,
                               maximizing_player_index,
                               child_state.current_player,
                               alpha, beta, log_tree, child_node)
            child_node['move'] = str(move) if move else 'Draw'
            tree_node['children'].append(child_node)

            if score > best_score:
                best_score = score
                best_move = move
            alpha = max(alpha, best_score)
            if beta <= alpha:
                break  # Beta cutoff
        tree_node['score'] = best_score
        return best_score, best_move
    else:
        # MIN node (opponent)
        tree_node['type'] = 'MIN'
        worst_score = math.inf
        for move in moves:
            child_state = apply_move(state, move)
            child_node = {}
            score, _ = minimax(child_state, depth - 1,
                               maximizing_player_index,
                               child_state.current_player,
                               alpha, beta, log_tree, child_node)
            child_node['move'] = str(move) if move else 'Draw'
            tree_node['children'].append(child_node)

            if score < worst_score:
                worst_score = score
                best_move = move
            beta = min(beta, worst_score)
            if beta <= alpha:
                break  # Alpha cutoff
        tree_node['score'] = worst_score
        return worst_score, best_move


def minimax_decision(state, player_index, depth=3):
    """
    Returns the best move for the minimax player and prints all considered moves.
    """
    hand = state.hands[player_index]
    valid_moves = get_valid_moves(hand, state.top_card)
    moves = valid_moves if valid_moves else [None]

    print(f"\n  [Minimax P{player_index+1}] Top card: {state.top_card}")
    print(f"  Hand: {hand}")
    print(f"  AI decision (all possible decisions at depth {depth}):")

    best_move = None
    best_score = -math.inf
    tree_root = {'children': []}

    for move in moves:
        child_state = apply_move(state, move)
        child_node = {}
        score, _ = minimax(child_state, depth - 1,
                           player_index,
                           child_state.current_player,
                           tree_node=child_node)
        child_node['move'] = str(move) if move else 'Draw'
        tree_root['children'].append(child_node)

        label = str(move) if move else 'Draw a card'
        print(f"    Play: {label:<20} Expected score: {score:.2f}")

        if score > best_score:
            best_score = score
            best_move = move

    print(f"  >>> Best move: {best_move if best_move else 'Draw'} (score: {best_score:.2f})")
    return best_move, tree_root

## 8. Expectimax Algorithm (Offensive – Player 2)

In [ ]:
def expectimax(state, depth, ai_player_index, current_turn_index, tree_node=None):
    """
    Expectimax algorithm.
    
    Node types:
      MAX node   → AI's turn (player_index == ai_player_index)
      CHANCE node → draw card action (when no valid moves exist)
      OPPONENT   → random legal move chosen with uniform probability
    """
    if tree_node is None:
        tree_node = {}

    if depth == 0 or state.is_terminal():
        val = evaluate(state, ai_player_index, 'offensive')
        tree_node['score'] = val
        tree_node['type'] = 'leaf'
        return val, None

    hand = state.hands[current_turn_index]
    valid_moves = get_valid_moves(hand, state.top_card)

    tree_node['player'] = f'P{current_turn_index + 1}'
    tree_node['top_card'] = str(state.top_card)
    tree_node['hand_size'] = len(hand)
    tree_node['children'] = []

    if current_turn_index == ai_player_index:
        # MAX node
        tree_node['type'] = 'MAX'

        if not valid_moves:
            # Must draw — this is a CHANCE node
            return _chance_node(state, depth, ai_player_index, current_turn_index, tree_node)

        best_score = -math.inf
        best_move = None
        for move in valid_moves:
            child_state = apply_move(state, move)
            child_node = {}
            score, _ = expectimax(child_state, depth - 1,
                                  ai_player_index,
                                  child_state.current_player,
                                  child_node)
            child_node['move'] = str(move)
            tree_node['children'].append(child_node)
            if score > best_score:
                best_score = score
                best_move = move

        # Also consider drawing voluntarily (though AI prefers to play)
        chance_node = {}
        draw_score = _chance_node_value(state, depth, ai_player_index,
                                        current_turn_index, chance_node)
        chance_node['move'] = 'Draw (chance)'
        tree_node['children'].append(chance_node)

        if draw_score > best_score:
            best_score = draw_score
            best_move = None  # None = draw

        tree_node['score'] = best_score
        return best_score, best_move

    else:
        # OPPONENT node — treat as random (uniform probability over legal moves)
        tree_node['type'] = 'OPPONENT'
        moves = valid_moves if valid_moves else [None]
        prob = 1.0 / len(moves)
        expected = 0.0

        for move in moves:
            child_state = apply_move(state, move)
            child_node = {}
            score, _ = expectimax(child_state, depth - 1,
                                  ai_player_index,
                                  child_state.current_player,
                                  child_node)
            child_node['move'] = str(move) if move else 'Draw'
            child_node['prob'] = prob
            tree_node['children'].append(child_node)
            expected += prob * score

        tree_node['score'] = expected
        return expected, None


def _chance_node(state, depth, ai_player_index, current_turn_index, tree_node):
    """Handles mandatory draw as a CHANCE node."""
    val = _chance_node_value(state, depth, ai_player_index, current_turn_index, tree_node)
    return val, None


def _chance_node_value(state, depth, ai_player_index, current_turn_index, tree_node):
    """
    CHANCE node: calculates expected value of drawing from deck.
    Each card in the remaining deck is equally likely to be drawn.
    """
    tree_node['type'] = 'CHANCE'
    tree_node['children'] = []

    if not state.deck:
        # No deck, evaluate as-is
        val = evaluate(state, ai_player_index, 'offensive')
        tree_node['score'] = val
        return val

    # Group deck cards to avoid computing identical futures multiple times
    unique_cards = list({(c.color, c.value): c for c in state.deck}.values())
    n = len(state.deck)
    expected = 0.0

    for card in unique_cards:
        # Probability this card is drawn
        count = sum(1 for c in state.deck if c.color == card.color and c.value == card.value)
        prob = count / n

        # Simulate drawing this card
        sim_state = state.clone()
        sim_state.hands[current_turn_index].append(card)
        # Remove one copy from deck
        for i, c in enumerate(sim_state.deck):
            if c.color == card.color and c.value == card.value:
                sim_state.deck.pop(i)
                break
        sim_state.current_player = (current_turn_index + 1) % 3

        child_node = {}
        score, _ = expectimax(sim_state, depth - 1,
                              ai_player_index,
                              sim_state.current_player,
                              child_node)
        child_node['move'] = f'Draw {card}'
        child_node['prob'] = prob
        tree_node['children'].append(child_node)
        expected += prob * score

    tree_node['score'] = expected
    return expected


def expectimax_decision(state, player_index=1, depth=3):
    """
    Returns the best move for the Expectimax player and prints all considered moves.
    """
    hand = state.hands[player_index]
    valid_moves = get_valid_moves(hand, state.top_card)
    moves = valid_moves if valid_moves else [None]

    print(f"\n  [Expectimax P{player_index+1}] Top card: {state.top_card}")
    print(f"  Hand: {hand}")
    print(f"  AI decision (all possible decisions at depth {depth}):")

    best_move = None
    best_score = -math.inf
    tree_root = {'children': [], 'type': 'ROOT'}

    for move in moves:
        if move is None:
            # Chance node for draw
            child_node = {}
            score = _chance_node_value(state, depth - 1, player_index,
                                       player_index, child_node)
            child_node['move'] = 'Draw (chance node)'
        else:
            child_state = apply_move(state, move)
            child_node = {}
            score, _ = expectimax(child_state, depth - 1,
                                  player_index,
                                  child_state.current_player,
                                  child_node)
            child_node['move'] = str(move)

        tree_root['children'].append(child_node)
        label = str(move) if move else 'Draw a card (chance)'
        print(f"    Play: {label:<25} Expected score: {score:.2f}")

        if score > best_score:
            best_score = score
            best_move = move

    print(f"  >>> Best move: {best_move if best_move else 'Draw'} (score: {best_score:.2f})")
    return best_move, tree_root

## 9. Game Tree Printer

In [ ]:
def print_tree(node, prefix="", is_last=True, max_depth=3, current_depth=0):
    """
    Recursively prints the search tree in a readable ASCII format.
    """
    if current_depth > max_depth:
        return

    connector = "└── " if is_last else "├── "
    move_label = node.get('move', 'ROOT')
    node_type  = node.get('type', '?')
    score      = node.get('score', '?')
    player     = node.get('player', '')
    prob_str   = f" [p={node['prob']:.2f}]" if 'prob' in node else ""

    if isinstance(score, float):
        score_str = f"{score:.2f}"
    else:
        score_str = str(score)

    print(f"{prefix}{connector}[{node_type}]{prob_str} Move: {move_label} | Score: {score_str} {player}")

    children = node.get('children', [])
    new_prefix = prefix + ("    " if is_last else "│   ")

    for i, child in enumerate(children):
        print_tree(child, new_prefix, i == len(children) - 1,
                   max_depth, current_depth + 1)

## 10. Game Loop

In [ ]:
PLAYER_NAMES = ["P1 (Minimax-Defensive)", "P2 (Expectimax-Offensive)", "P3 (Minimax/User)"]


def print_state(state):
    print("\n" + "=" * 55)
    print(f"  Top Card : {state.top_card}")
    print(f"  Deck Size: {len(state.deck)} cards remaining")
    for i, hand in enumerate(state.hands):
        print(f"  {PLAYER_NAMES[i]}: {hand} ({len(hand)} cards)")
    print("=" * 55)


def get_user_move(hand, top_card):
    """
    Prompts the human player to choose a move.
    """
    valid = get_valid_moves(hand, top_card)
    print(f"\n  Your hand: {hand}")
    print(f"  Top card : {top_card}")

    if not valid:
        print("  No valid moves. You must draw a card.")
        return None

    print("  Valid moves:")
    for i, card in enumerate(valid):
        print(f"    [{i}] {card}")
    print(f"    [d] Draw a card")

    while True:
        choice = input("  Enter your choice (index or 'd'): ").strip().lower()
        if choice == 'd':
            return None
        try:
            idx = int(choice)
            if 0 <= idx < len(valid):
                return valid[idx]
        except ValueError:
            pass
        print("  Invalid input, try again.")


def play_game(mode='simulation', depth=3, max_turns=100):
    """
    Main game loop.
    mode: 'simulation' – all 3 players are AI
          'manual'     – P3 is human-controlled
    """
    print("\n" + "*" * 55)
    print("        SIMPLIFIED UNO — AI GAME")
    print(f"        Mode: {mode.upper()}")
    print("*" * 55)

    # --- Setup ---
    deck = generate_deck()
    hands = [deck[:5], deck[5:10], deck[10:15]]
    top_card = deck[15]
    remaining_deck = deck[16:]

    state = GameState(hands, top_card, remaining_deck, current_player=0)

    print("\n  Initial deal:")
    print_state(state)

    trees = {}  # Store first tree per player for display

    turn = 0
    while not state.is_terminal() and turn < max_turns:
        cp = state.current_player
        turn += 1
        print(f"\n{'─'*55}")
        print(f"  Turn {turn:>3} | {PLAYER_NAMES[cp]}'s move")
        print(f"{'─'*55}")

        if cp == 0:
            # Player 1 – Minimax Defensive
            move, tree = minimax_decision(state, player_index=0, depth=depth)
            if cp not in trees:
                trees[cp] = tree

        elif cp == 1:
            # Player 2 – Expectimax Offensive
            move, tree = expectimax_decision(state, player_index=1, depth=depth)
            if cp not in trees:
                trees[cp] = tree

        else:
            # Player 3
            if mode == 'manual':
                move = get_user_move(state.hands[2], state.top_card)
            else:
                # Simulation – reuse Minimax logic
                move, tree = minimax_decision(state, player_index=2, depth=depth)
                if cp not in trees:
                    trees[cp] = tree

        # Apply the chosen move
        move_label = str(move) if move else 'Draw a card'
        print(f"\n  *** {PLAYER_NAMES[cp]} plays: {move_label} ***")
        state = apply_move(state, move)
        print_state(state)

    # --- Game Over ---
    print("\n" + "*" * 55)
    if state.is_terminal():
        winner = state.winner()
        print(f"  GAME OVER!  Winner: {PLAYER_NAMES[winner]}")
    else:
        print("  GAME OVER! Max turns reached — no winner.")
    print("*" * 55)

    return state, trees

## 11. Run Simulation Mode

In [ ]:
random.seed(42)   # reproducible results
final_state, game_trees = play_game(mode='simulation', depth=3, max_turns=60)

## 12. Generated Search Trees

In [ ]:
print("\n" + "="*55)
print("  SEARCH TREE — P1 Minimax (Defensive)")
print("="*55)
if 0 in game_trees:
    print_tree(game_trees[0], max_depth=2)
else:
    print("  (tree not captured)")

In [ ]:
print("\n" + "="*55)
print("  SEARCH TREE — P2 Expectimax (Offensive)")
print("="*55)
if 1 in game_trees:
    print_tree(game_trees[1], max_depth=2)
else:
    print("  (tree not captured)")

In [ ]:
print("\n" + "="*55)
print("  SEARCH TREE — P3 Minimax (Simulation Mode)")
print("="*55)
if 2 in game_trees:
    print_tree(game_trees[2], max_depth=2)
else:
    print("  (tree not captured)")

## 13. Manual Mode Demo (P3 as Human)

In [ ]:
# Uncomment the line below to play manually as P3
# random.seed(7)
# play_game(mode='manual', depth=3, max_turns=60)

## 14. Evaluation Function — Deep Explanation

### Base Formula
$$\text{Score} = 50 - 5 \cdot C_{AI} + 2 \cdot C_{opp} + 3 \cdot S$$

| Symbol | Meaning |
|--------|---------|
| $C_{AI}$ | Number of cards in the current AI's hand |
| $C_{opp}$ | Average cards held by the two opponents |
| $S$ | Number of Skip cards in AI's hand |

### Why This Formula?

- **$50$ (base):** Provides a positive baseline so scores near neutral are easy to interpret.
- **$-5 \cdot C_{AI}$:** Each card you hold is a liability — getting closer to zero cards means you are closer to winning. The coefficient **5** makes your own card count the dominant factor.
- **$+2 \cdot C_{opp}$:** Opponents having more cards is good for you — they are further from winning. Weight **2** is intentionally smaller than your own penalty so the AI prioritizes self-improvement over slowing opponents.
- **$+3 \cdot S$:** Skip cards let you deny an opponent their entire turn, which is strategically very powerful. Weight **3** rewards holding them until the right moment.

### Defensive Tuning (P1 – Minimax)
```
w_ai = 6, w_opp = 3, w_skip = 5
```
- **Higher $w_{ai} = 6$**: P1 is extremely cautious about its own hand — it avoids risky plays that could make it draw.
- **Higher $w_{opp} = 3$**: P1 actively values keeping opponents loaded with cards. Combined with skip management, this is pure defense.
- **Highest $w_{skip} = 5$**: Skips are P1's primary weapon. It hoards them and plays at critical moments to reset opponents' momentum.

### Offensive Tuning (P2 – Expectimax)
```
w_ai = 4, w_opp = 1, w_skip = 2
```
- **Lower $w_{ai} = 4$**: P2 is willing to take more risks — it may play a card even if drawing is slightly unfavorable, because its goal is to empty its hand fast.
- **Lower $w_{opp} = 1$**: P2 is largely indifferent to opponent hand sizes. It focuses on its own card shedding.
- **Lower $w_{skip} = 2$**: P2 treats skips as normal cards to shed, not to hoard.

### Terminal State Handling
- **Win:** $+1000$ bonus (guarantees winning states are always preferred)
- **Loss:** $-1000$ penalty (guarantees losing states are avoided if any alternative exists)


## 15. Algorithm Comparison — Minimax vs Expectimax

### Strategies Employed

| Aspect | Minimax (P1, Defensive) | Expectimax (P2, Offensive) |
|--------|------------------------|---------------------------|
| **Core idea** | Assumes opponents play optimally against you (worst case) | Treats opponents as random and draws as probabilistic chance nodes |
| **Pruning** | Alpha-Beta pruning for efficiency | No pruning (needs full probability distribution) |
| **Draw handling** | Draw = MIN opponent punishes you | Draw = CHANCE node with expected value over deck |
| **Strategy** | Defensive, risk-averse, Skip-hoarding | Aggressive, fast card shedding, probabilistic |
| **Depth** | 3 | 3 |

### Which Algorithm Performs Best?

**In this simplified UNO setting, Expectimax (P2) tends to perform better overall** for the following reasons:

1. **Opponent modeling accuracy**: Minimax assumes opponents always play *optimally against you*. In reality (and in this implementation), opponents do not coordinate against you — each player plays for themselves. Expectimax's random-opponent model is closer to reality.

2. **Draw card expected value**: Expectimax explicitly calculates the probability of drawing a useful card, allowing it to make informed decisions about whether drawing is worth it. Minimax treats drawing pessimistically (MIN node), potentially undervaluing draws when the deck is rich with useful cards.

3. **Offensive efficiency**: Because UNO is won by emptying your hand, aggressive shedding (P2's goal) is directly aligned with the win condition. Defensive play (P1's goal) prolongs the game but doesn't directly win it.

4. **Skip management trade-off**: P1 hoards Skips, but holding cards is a liability. P2 discards Skips opportunistically, which often results in fewer cards held on average.

### Caveat
Minimax with Alpha-Beta is computationally more efficient (fewer nodes evaluated) than Expectimax, and in highly adversarial or deterministic settings, Minimax would be preferred. In a card game with chance elements and non-cooperative opponents, Expectimax is the more appropriate model.
